# 01 — Point Representation

Before any algorithm, we need to establish one rule absolutely clearly. This is the rule that will silently break everything if you miss it:

```text
ipyleaflet click events  →  [lat, lon]   (latitude first)
GeoJSON coordinates      →  [lon, lat]   (longitude first)
```

They are backwards from each other. Always. Without exception.

If you get this wrong, your points will appear in the wrong ocean and your polygon tests will silently return false for everything. The bug is invisible until you look at the coordinates on a map — at which point you'll see your click near the coast of Somalia when you clicked on Iran.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, Marker, basemaps
from ipywidgets import Output

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Ready.")

## The Coordinate Order Problem

Why do two tools in the same ecosystem use opposite conventions?

**ipyleaflet** follows Leaflet.js, which follows cartographic convention: latitude (north/south) first, longitude (east/west) second. This is how addresses are written, how GPS reads out, and how most mapping interfaces work.

**GeoJSON** follows mathematical convention: x-axis (longitude, east/west) first, y-axis (latitude, north/south) second. GeoJSON coordinates are `[x, y]` = `[lon, lat]`.

Both conventions are internally consistent. They just disagree with each other. And you are always in the middle.

```python
# ipyleaflet — kwargs['coordinates'] from a click:
coords = [35.695, 51.388]   # [lat, lon] — Tehran

# GeoJSON Point geometry:
geojson_point = {"type": "Point", "coordinates": [51.388, 35.695]}  # [lon, lat]

# Marker location:
marker = Marker(location=(35.695, 51.388))   # (lat, lon) — same as ipyleaflet
```

In [ ]:
# Demonstrate the flip with Tehran's known coordinates
tehran_lat, tehran_lon = 35.695, 51.388

# What ipyleaflet click gives you:
click_coords = [tehran_lat, tehran_lon]   # [lat, lon]
print(f"From click event:  {click_coords}  (lat first)")

# What GeoJSON needs:
geojson_coords = [tehran_lon, tehran_lat]  # [lon, lat]
print(f"For GeoJSON:       {geojson_coords}  (lon first)")

# The extraction pattern — always the same:
lat = click_coords[0]
lon = click_coords[1]
print(f"\nExtracted:  lat={lat}  lon={lon}")

## Creating a GeoJSON Point Feature

Once we have a click, we want to store it as a proper GeoJSON Point feature — compatible with everything else in the course.

In [ ]:
def click_to_point_feature(click_coords, properties=None):
    """
    Convert ipyleaflet click coordinates [lat, lon]
    to a GeoJSON Point Feature with [lon, lat] coordinates.
    """
    lat, lon = click_coords[0], click_coords[1]
    return {
        "type": "Feature",
        "properties": properties or {},
        "geometry": {
            "type": "Point",
            "coordinates": [lon, lat]   # GeoJSON: lon first
        }
    }

# Test with Tehran
fake_click = [35.695, 51.388]   # as if from kwargs['coordinates']
feature = click_to_point_feature(fake_click, properties={"label": "Tehran"})
print(json.dumps(feature, indent=2))

## Adding a Marker at Click Location

`Marker` uses `(lat, lon)` — same order as ipyleaflet events — so no flip needed there. But `GeoJSON` and any polygon test needs `[lon, lat]`.

In [ ]:
markers = []   # keep references so they can be removed
clicked_features = []

out = Output()
m = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def on_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']   # [lat, lon]
        lat, lon = coords[0], coords[1]

        # Add a Marker (uses lat, lon directly)
        marker = Marker(location=(lat, lon))
        m.add(marker)
        markers.append(marker)

        # Also store as GeoJSON feature (flip to lon, lat)
        feat = click_to_point_feature(coords, {"n": len(markers)})
        clicked_features.append(feat)

        with out:
            out.clear_output(wait=True)
            print(f"{len(markers)} marker(s) placed:")
            for f in clicked_features:
                c = f['geometry']['coordinates']
                print(f"  #{f['properties']['n']}  lon={c[0]:.4f}  lat={c[1]:.4f}  [GeoJSON order]")

m.on_interaction(on_click)
display(m, out)

## Saving Clicked Points to a File

After clicking, we can persist the collected points as a GeoJSON FeatureCollection for use in later notebooks.

In [ ]:
# Run after clicking the map above
if clicked_features:
    fc = {"type": "FeatureCollection", "features": clicked_features}
    out_path = DATA_DIR / "clicked_points.geojson"
    with open(out_path, "w") as f:
        json.dump(fc, f, indent=2)
    print(f"Saved {len(clicked_features)} point(s) → {out_path}")
else:
    print("No clicks yet — use the map above first, then re-run this cell.")

## The Rule — Written Down

Put this somewhere visible and read it every time:

```python
# From a click event:
lat = kwargs['coordinates'][0]   # index 0 = lat
lon = kwargs['coordinates'][1]   # index 1 = lon

# For GeoJSON geometry:
"coordinates": [lon, lat]        # lon first

# For Marker location:
Marker(location=(lat, lon))      # lat first

# For polygon test (our PIP function, next notebook):
point_in_polygon(lon, lat, ring) # we'll use lon, lat consistently
```

The pattern: **always extract `lat, lon` from clicks immediately, then use `lon, lat` everywhere geometric**.

## Exercise A

Write a function `format_point(click_coords)` that takes ipyleaflet click coordinates and returns a GeoJSON Point Feature.

1. The function must include a `"click_index"` property that you pass in as a second argument.
2. Test it with these manually-specified fake click coordinates:
   - `[35.695, 51.388]` → Tehran
   - `[24.688, 46.675]` → Riyadh
   - `[40.417, -3.703]` → Madrid
3. Print the GeoJSON coordinates for each and verify the lon/lat order is correct.

In [ ]:
import json

def format_point(click_coords, click_index):
    lat, lon = click_coords
    return {
        'type': 'Feature',
        'properties': {'click_index': click_index},
        'geometry': {'type': 'Point', 'coordinates': [lon, lat]},
    }

test_clicks = [
    ([35.695, 51.388], 'Tehran'),
    ([24.688, 46.675], 'Riyadh'),
    ([40.417, -3.703], 'Madrid'),
]

for i, (coords, name) in enumerate(test_clicks):
    feat = format_point(coords, i + 1)
    print(f"{name:<8} -> {feat['geometry']['coordinates']}")

## Exercise B

Create a map that places a **Marker** at each click and simultaneously adds the click as a **GeoJSON Point** layer.

1. The Marker and the GeoJSON point must appear at the exact same location.
2. After 3 clicks, print the GeoJSON coordinates of all stored points.
3. Verify each point's `coordinates` field is in `[lon, lat]` order (longitude first).

In [ ]:
from ipyleaflet import Map, GeoJSON, Marker, basemaps
from ipywidgets import Output

clicked_features = []
point_layers = []
out_clicks = Output()
m_clicks = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def on_click(**kwargs):
    if kwargs.get('type') != 'click':
        return
    coords = kwargs['coordinates']
    lat, lon = coords
    marker = Marker(location=(lat, lon))
    m_clicks.add(marker)
    point_layers.append(marker)

    feat = format_point(coords, len(clicked_features) + 1)
    clicked_features.append(feat)
    gj = GeoJSON(data={'type': 'FeatureCollection', 'features': [feat]}, point_style={'radius': 7, 'color': '#e63946', 'fillOpacity': 0.9})
    m_clicks.add(gj)
    point_layers.append(gj)

    with out_clicks:
        out_clicks.clear_output(wait=True)
        print(f'{len(clicked_features)} point(s) stored.')
        if len(clicked_features) >= 3:
            print('GeoJSON coordinates after 3 clicks:')
            for feat in clicked_features:
                print(f"  {feat['geometry']['coordinates']}")

m_clicks.on_interaction(on_click)
display(m_clicks, out_clicks)

---

## Check Your Understanding

**1.** Why does ipyleaflet use `[lat, lon]` while GeoJSON uses `[lon, lat]`?

**2.** What would happen if you passed click coordinates directly to a GeoJSON geometry without flipping them?

```python
# No code needed — answer in your own words
```

## Next

In [02 — Point in Polygon Basics](./02_Point_In_Polygon_Basics.ipynb), we build the intuition for what "inside a polygon" means before touching any algorithm — covering convex shapes, concave shapes, and the edge cases that cause headaches.